based on cca_simulation2, I want to see if using cross validation i can obtain better weight estimates (more similar ot the gt weights) using less trials.

- My hyperparameter is the L2 regularization term
- My parameters are the time lag and the weights

i am using ridge regression and minimizing the mse to select lambda, but selecting lag and weights maximizing the correlaiton

In [1]:
from cca_simulation2_script import *

In [ ]:
# ==== add below your helpers in cca_simulation2_script.py =====================

from dataclasses import dataclass
from typing import List, Tuple, Dict
import numpy as np
  
def softmax(w: np.ndarray, tau: float = 1.0) -> np.ndarray:
    z = (w / max(tau, 1e-12)).astype(float)
    z -= z.max()                   # numerical stability
    e = np.exp(z)
    return e / (e.sum() + 1e-12)

def ridge_wx(X: np.ndarray, y: np.ndarray, lam: float, eps: float = 1e-12, normalise: bool = False ) -> np.ndarray:
    """
    Return ridge-regression weights w for y ≈ Xw.

    Same covariance-style formulation as before, but:
    * no softmax
    * optional L2 normalisation if normalise=True (default False).
    """
    # centre like np.cov does
    Xc = X - X.mean(axis=0, keepdims=True)
    yc = y - y.mean()
    n  = Xc.shape[0]
    Sxx = (Xc.T @ Xc) / max(n - 1, 1)      # ddof=1 covariance
    Sxy = (Xc.T @ yc) / max(n - 1, 1)

    A = Sxx + lam * np.eye(Sxx.shape[0])
    w = np.linalg.solve(A, Sxy)

    return w
    

def corr_with_weights(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> float:
    u = X @ w
    # y is already z-scored per trial in concat_trials; u doesn't need scaling for corr
    if u.std(ddof=0) == 0 or y.std(ddof=0) == 0:
        return np.nan
    return float(np.corrcoef(u, y)[0, 1])

def mse_with_weights(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> float:
    """
    Mean squared error between y and Xw.
    Assumes X, y already preprocessed by concat_trials.
    """
    y_hat = X @ w
    return float(np.mean((y_hat - y) ** 2))


# ---------------- per-(lag, λ) training step ---------------------------------
def fit_at_shift_lambda(trials: List[dict], shift_samples: int, lam: float, trim_ms: int, normalise: bool = True) -> Tuple[np.ndarray, float]:
    """
    Fit weights on concatenated TRAIN trials aligned with the given shift.
    Returns (w, train_mse, r_train).
    """
    X_tr, y_tr = concat_trials(trials, shift_samples=shift_samples, trim_ms=trim_ms)

    if len(y_tr) == 0:
        return np.zeros(trials[0]['X'].shape[1]), np.nan
    #w_zoo = ridge_cca_wx_zoo(X_tr, y_tr, lam)

    w_ridge = ridge_wx(X_tr, y_tr, lam, normalise=False)
    train_mse = mse_with_weights(X_tr, y_tr, w_ridge)

    r_train = corr_with_weights(X_tr, y_tr, w_ridge)
    return w_ridge, train_mse, r_train

def evaluate_on_trials(trials: List[dict], shift_samples: int, w: np.ndarray, trim_ms: int) -> float:
    """Correlation on a (TRAIN or TEST) split using fixed shift and weights."""
    X_te, y_te = concat_trials(trials, shift_samples=shift_samples, trim_ms=trim_ms)
    if len(y_te) == 0:
        return np.nan
    #return corr_with_weights(X_te, y_te, w)
    return mse_with_weights(X_te, y_te, w), corr_with_weights(X_te, y_te, w)

def triplet_splits_from_trials(trials: List[dict], seed: int = 7) -> List[Tuple[np.ndarray, np.ndarray]]:
    """
    Leave-one-triplet-out using the 'triplet_id' present in each trial.
    Returns a list of (train_idx, test_idx) pairs; each test_idx has 3 indices.
    """
    import numpy as np
    rng = np.random.default_rng(seed)

    # map triplet_id -> list of trial indices
    id_to_idx = {}
    for i, tr in enumerate(trials):
        tid = tr.get('triplet_id')
        if tid is None:
            raise ValueError("Trial missing 'triplet_id'; ensure simulator sets it.")
        id_to_idx.setdefault(tid, []).append(i)

    trip_ids = list(id_to_idx.keys())
    rng.shuffle(trip_ids)

    splits = []
    all_idx = np.arange(len(trials))
    for tid in trip_ids:
        test_idx = np.array(sorted(id_to_idx[tid]), dtype=int)  # exactly 3
        train_idx = np.setdiff1d(all_idx, test_idx, assume_unique=False)
        splits.append((train_idx, test_idx))
    return splits



# ---------------- CV harness --------------------------------------------------
@dataclass
class CVResult:
    lam: float
    mean_train_r: float
    mean_test_r: float
    mean_train_mse: float
    mean_test_mse: float
    per_fold: List[Dict]            # diagnostics

def kfold_indices(n_trials: int, k: int, rng: np.random.Generator) -> List[Tuple[np.ndarray, np.ndarray]]:
    idx = np.arange(n_trials)
    rng.shuffle(idx)
    folds = np.array_split(idx, k)
    splits = []
    for f in range(k):
        test_idx = folds[f]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != f])
        splits.append((train_idx, test_idx))
    return splits

from typing import Optional

def cross_validate_l2_and_lag(
    trials: List[dict],
    lambdas: List[float],
    candidate_shifts: np.ndarray,
    trim_ms: int = 200,          # safety edge-trim (10 ms samples → 20 samples)
    k_folds: int = 4,
    seed: int = 7,
    splits: Optional[List[Tuple[np.ndarray, np.ndarray]]] = None,  # may be LOTO triplets
    print_splits: bool = False,   # NEW: print split diagnostics
    normalise: bool = True
) -> Tuple[List[CVResult], Dict]:
    """
    If 'splits' is provided, use those (train_idx, test_idx) pairs directly.
    Otherwise fall back to k-fold splits created from k_folds & seed.

    Returns:
      ordered_results: list[CVResult] sorted by mean_test_mse (asc, tie→lower λ)
      final_fit: {'lambda','best_shift','w','train_mse_all','train_rs_all',
                  'cv_mean_test_mse','cv_mean_test_r',
                  'cv_mean_train_mse','cv_mean_train_r','cv_details'}
    """
    rng = np.random.default_rng(seed)
    if splits is None:
        splits = kfold_indices(len(trials), k_folds, rng)  # old behavior

    # ---- Split diagnostics ----
    if print_splits:
        print(f"[CV] Using {len(splits)} folds")
        for f, (_, test_idx) in enumerate(splits, start=1):
            test_idx = np.array(test_idx, dtype=int)
            trip_ids = [trials[i].get('triplet_id', None) for i in test_idx]
            lens     = [trials[i].get('length_s', None)   for i in test_idx]
            uniq_trip = set([t for t in trip_ids if t is not None])
            msg_trip  = f"triplet_id(s)={sorted(list(uniq_trip))}" if any(t is not None for t in trip_ids) else "triplet_id(s)=<n/a>"
            msg_len   = f"length_s={lens}" if any(L is not None for L in lens) else "length_s=<n/a>"
            print(f"  - Fold {f:02d}: test_idx={test_idx.tolist()} | {msg_trip} | {msg_len} | n_test={len(test_idx)}")
            # quick sanity checks (non-fatal)
            if len(test_idx) == 3 and len(uniq_trip) == 1:
                print("    ✓ looks like a single triplet (3 trials, same triplet_id)")
            elif len(test_idx) == 3:
                print("    ! 3 trials but multiple/unknown triplet_id; check simulator tags.")
            else:
                print("    ! test size != 3; this is fine for standard k-fold but not LOTO-triplet.")

    # ---- CV across lambdas -------------------------------------------
    cv_summaries: List[CVResult] = []

    for lam in lambdas:
        print(f"[CV] Evaluating λ factor {lam:.6g} out of {len(lambdas)}")
        fold_rows = []
        train_rss, test_rss = [], []
        train_mses, test_mses = [], []

        for train_idx, test_idx in splits:
            train_trials = [trials[i] for i in train_idx]
            test_trials  = [trials[i] for i in test_idx]

            # 1) search best lag on TRAIN for this λ (MAXIMISE train correlation)
            best_shift, best_train_mse, rs_best, best_w = None, np.inf, -np.inf, None
            for s in candidate_shifts:
                w_s, mse_tr, rs_tr = fit_at_shift_lambda(train_trials, int(s), lam, trim_ms, normalise=normalise)
                
                if np.isfinite(rs_tr) and rs_tr > rs_best:
                    best_train_mse, best_shift, rs_best, best_w = mse_tr, int(s), rs_tr, w_s

            # 2) evaluate on TEST using best (lag, w)
            mse_te, rs_te = evaluate_on_trials(test_trials, best_shift, best_w, trim_ms)

            fold_rows.append({
                "lam": lam, "shift": best_shift,
                "train_mse": best_train_mse, "test_mse": mse_te,
                "r_train": rs_best, "r_test": rs_te,
                "n_train": len(train_idx), "n_test": len(test_idx),
            })
            train_rss.append(rs_best);  test_rss.append(rs_te)
            train_mses.append(best_train_mse); test_mses.append(mse_te)

        cv_summaries.append(CVResult(
            lam=lam,
            mean_train_r=float(np.nanmean(train_rss)),
            mean_test_r=float(np.nanmean(test_rss)),
            mean_train_mse=float(np.nanmean(train_mses)),
            mean_test_mse=float(np.nanmean(test_mses)),
            per_fold=fold_rows,
        ))

    # pick λ with LOWEST mean TEST MSE (tie-breaker: smaller λ)
    ordered_results = sorted(cv_summaries, key=lambda c: (c.mean_test_mse, c.lam))
    best = ordered_results[0]

    # ------- recompute DEFINITIVE lag + weights at the chosen λ on ALL trials
    best_shift_all, best_mse_all, best_rs_all, best_w_all = None, np.inf, -np.inf, None
    for s in candidate_shifts:
        w_s, mse_all, rs_all = fit_at_shift_lambda(
            trials, int(s), best.lam, trim_ms, normalise=normalise
        )
        if np.isfinite(rs_all) and rs_all > best_rs_all:
            best_mse_all, best_shift_all, best_rs_all, best_w_all = mse_all, int(s), rs_all, w_s


    final_fit = {
        "lambda": best.lam,
        "best_shift": best_shift_all,
        "w": best_w_all,
        "train_mse_all": best_mse_all,
        "train_rs_all": best_rs_all,
        "cv_mean_test_mse": best.mean_test_mse,
        "cv_mean_test_r": best.mean_test_r,
        "cv_mean_train_mse": best.mean_train_mse,
        "cv_mean_train_r": best.mean_train_r,
        "cv_details": best.per_fold,
    }
    return ordered_results, final_fit


with slow component

In [ ]:
# ===================== SWEEP OVER k (uses the helpers you pasted) =====================
import numpy as np
import pandas as pd

if __name__ == "__main__":
    EXPT = 20251113                     # ← change this to make a new experiment
    #EXPT = 7
    global RNG
    RNG   = np.random.default_rng(EXPT) # global fallback

    mode = "slow"
    
    rng_w0 = np.random.default_rng(EXPT + 1)   # new w0 each experiment
    #rng_w0 = np.random.default_rng(42)
    TRUE_LAG_MS = -350
    SHIFTS = candidate_lags_units(step_ms=10, max_ms=1500)
    W_SPREAD = 0.7

    #LAMBDAS = [0, 0.001, 0.01, 0.1, 1, 5, 10, 100, 1000, 100000]
    LAMBDAS = np.logspace(-3, 1.5, 15)
    TRIM_MS = 200

    # Fix GT weights ONCE for comparable cosine similarity across k
    w0 = 1.0 + W_SPREAD * rng_w0.normal(size=N_CH)
    #w0 = w0 / (np.linalg.norm(w0) or 1.0)

    # k in {4, 9, 12, 15, ..., 39}
    #ks = [4] + list(range(9, 20, 3))
    n_triplets = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 17, 20]

    rows = []
    for k in n_triplets:
        print(f"\n--- CV sweep at k={k} folds ---")
        # simulate 3*k trials (3 trials per test fold)
        sim_rng = np.random.default_rng(EXPT * 1_000 + k)  # unique per k & experiment
        #sim_rng = np.random.default_rng(10_000 + k)
        trials_og = simulate_component_dataset(
            n_triplets=k, triplet_lengths_s=(26.0, 18.0, 10.0),
            pupil_lag_ms=TRUE_LAG_MS, w_true_fixed=w0, rng=sim_rng,
            pupil_extra='ar1', pupil_extra_scale=0.5,
            lag_jitter_ms=abs(TRUE_LAG_MS)*0.25,
            eeg_weight_drift=True, eeg_noise_sd=0.2
        )

        # ----- CV to choose λ* -----
        splits = triplet_splits_from_trials(trials_og, seed=7)
        trials_comp = split_components_all_trials(trials_og, fs=FS, low_fc=0.25, band=(0.4, 0.7))

        if mode == "slow":
            trials = [{'X': tr['X_slow'], 'y': tr['y_slow'], 'w_true': tr['w_true'], 'c': tr['c']} for tr in trials_comp]
        elif mode == "fast":
            trials = [{'X': tr['X_fast'], 'y': tr['y_fast'], 'w_true': tr['w_true'], 'c': tr['c']} for tr in trials_comp]
        else:
            trials = trials_og

        ordered_cv, final_fit = cross_validate_l2_and_lag(
            trials=trials,
            lambdas=LAMBDAS,
            candidate_shifts=SHIFTS,
            trim_ms=TRIM_MS,
            k_folds=k, seed=7,
            splits=splits
        )

        order_kx = pd.DataFrame(ordered_cv)
        order_kx.drop(columns=['per_fold'], inplace=True)
        order_kx.to_csv(f"order_ridge_SLOW_k{k}.csv", index=False)

        lam_star = final_fit["lambda"]
        best_shift_star = final_fit["best_shift"]
        lag_ms_star = int(best_shift_star) * 10
        w_star = final_fit["w"]
        train_mse_all_star = final_fit["train_mse_all"]
        mean_test_mse_star = final_fit["cv_mean_test_mse"]
        train_rs_all_star = final_fit["train_rs_all"]
        mean_test_rs_star = final_fit["cv_mean_test_r"]

        # cosine similarity at λ*
        num = float(np.dot(w0, w_star))
        den = (np.linalg.norm(w0) or 1.0) * (np.linalg.norm(w_star) or 1.0)
        cos_star = abs(num / den)

        # ===== λ = 0 (no reg) — DO NOT use CV to choose lag/weights =====
        # Choose lag+weights on ALL trials (search best lag over SHIFTS)
        best_mse0_all, rs_best_mse0, best_s0, w0_all = np.inf, -np.inf, None, None
        for s in SHIFTS:
            w_s, mse_all, rs_all = fit_at_shift_lambda(trials, int(s), 0.0, TRIM_MS)
            if np.isfinite(rs_all) and rs_all > rs_best_mse0:
                best_mse0_all, rs_best_mse0, best_s0, w0_all = mse_all, rs_all, int(s), w_s


        lag_ms_zero = int(best_s0) * 10
        train_mse_all_zero = float(best_mse0_all)
        train_rs_all_zero = float(rs_best_mse0)

        # For fairness, compute MEAN TEST corr for λ=0 by *evaluating* this fixed (lag,w) on the folds,
        # but not using any CV information to select them.
        rng = np.random.default_rng(7)
        splits = kfold_indices(len(trials), k, rng)
        test_mses_zero, test_rs_zero = [], []
        for tr_idx, te_idx in splits:
            te_set = [trials[i] for i in te_idx]
            mse_te0, r_te0 = evaluate_on_trials(te_set, best_s0, w0_all, TRIM_MS)
            test_mses_zero.append(mse_te0)
            test_rs_zero.append(r_te0)
        mean_test_mse_zero = float(np.nanmean(test_mses_zero))
        mean_test_rs_zero = float(np.nanmean(test_rs_zero))

        # cosine similarity at λ=0
        num0 = float(np.dot(w0, w0_all))
        den0 = (np.linalg.norm(w0) or 1.0) * (np.linalg.norm(w0_all) or 1.0)
        cos_zero = abs(num0 / den0)

        rows.append({
            "k": k,
            "lam_star": lam_star,
            "best_lag_ms_star": lag_ms_star,
            "train_mse_all_star": float(train_mse_all_star),
            "mean_test_mse_star": float(mean_test_mse_star),
            "train_r_all_star": float(train_rs_all_star),
            "mean_test_r_star": float(mean_test_rs_star),
            "w_star": w_star,
            "cos_star": float(cos_star),
            "best_lag_ms_lam0": lag_ms_zero,
            "train_mse_all_lam0": train_mse_all_zero,
            "mean_test_mse_lam0": mean_test_mse_zero,
            "train_r_all_lam0": train_rs_all_zero,
            "mean_test_lam0": mean_test_rs_zero,
            "w_lam0": w0_all,
            "cos_lam0": float(cos_zero),
        })
        dfk_temp = pd.DataFrame(rows).sort_values("k").reset_index(drop=True)
        dfk_temp.to_csv("dfk_ridge_TMP.csv", index=False)

    # ---- results table
    dfk = pd.DataFrame(rows).sort_values("k").reset_index(drop=True)
    dfk.to_csv("dfk_ridge.csv", index=False)

    print("\n=== CV sweep (by k) ===")
    with pd.option_context('display.max_rows', None, 'display.width', 140):
        print(dfk.to_string(index=False, float_format=lambda x: f"{x:.3f}"))



--- CV sweep at k=3 folds ---
[CV] Evaluating λ factor 0.001 out of 15
[CV] Evaluating λ factor 0.00209618 out of 15
[CV] Evaluating λ factor 0.00439397 out of 15
[CV] Evaluating λ factor 0.00921055 out of 15
[CV] Evaluating λ factor 0.019307 out of 15
[CV] Evaluating λ factor 0.0404709 out of 15
[CV] Evaluating λ factor 0.0848343 out of 15
[CV] Evaluating λ factor 0.177828 out of 15
[CV] Evaluating λ factor 0.372759 out of 15
[CV] Evaluating λ factor 0.781371 out of 15
[CV] Evaluating λ factor 1.63789 out of 15
[CV] Evaluating λ factor 3.43332 out of 15
[CV] Evaluating λ factor 7.19686 out of 15
[CV] Evaluating λ factor 15.0859 out of 15
[CV] Evaluating λ factor 31.6228 out of 15

--- CV sweep at k=4 folds ---
[CV] Evaluating λ factor 0.001 out of 15
[CV] Evaluating λ factor 0.00209618 out of 15
[CV] Evaluating λ factor 0.00439397 out of 15
[CV] Evaluating λ factor 0.00921055 out of 15
[CV] Evaluating λ factor 0.019307 out of 15
[CV] Evaluating λ factor 0.0404709 out of 15
[CV] Evalu

In [9]:
import pandas as pd
import numpy as np
TRUE_LAG_MS = -350
dfk = pd.read_csv(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\simulated_check\simulation_ridge_reg\sim3\dfk_ridge_TMP.csv")

print("\n=== CV sweep (by k) ===")
with pd.option_context('display.max_rows', None, 'display.width', 140):
    print(dfk.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# ---- plots vs k
import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 3, figsize=(9, 4), sharex=True)

axs[0,0].plot(dfk["k"], dfk["lam_star"], marker="o")
axs[0,0].set_title("Optimal λ vs k"); axs[0,0].set_ylabel("λ*"); axs[0,0].grid(alpha=.3)

axs[0,2].plot(dfk["k"], dfk["best_lag_ms_star"], marker="o", label="λ*")
axs[0,2].plot(dfk["k"], dfk["best_lag_ms_lam0"], marker="s", ls="--", label="λ=0")
axs[0,2].set_title("Best lag vs k"); axs[0,2].set_ylabel(r"$\ell_{best} (ms)$"); axs[0,2].legend(); axs[0,2].grid(alpha=.3)

axs[1, 2].plot(dfk["k"], np.abs(TRUE_LAG_MS- dfk["best_lag_ms_star"]), marker="o", label="λ*")
axs[1, 2].plot(dfk["k"], np.abs(TRUE_LAG_MS- dfk["best_lag_ms_lam0"]), marker="s", ls="--", label="λ=0")
axs[1, 2].set_title("lag diff vs k"); axs[1, 2].set_ylabel(r"$|\ell_{gt} - \ell_{best}| (ms)$"); axs[1, 2].grid(alpha=.3)
axs[1,2].legend()
axs[0, 1].plot(dfk["k"], dfk["train_r_all_star"], marker="o", label="r λ*")
axs[0, 1].plot(dfk["k"], dfk["train_r_all_lam0"], marker="s", ls="--", label="r λ=0")
axs[0, 1].plot(dfk["k"], dfk["train_mse_all_star"], marker="o", label="MSE λ*")
axs[0, 1].plot(dfk["k"], dfk["train_mse_all_lam0"], marker="s", ls="--", label="MSE λ=0")
axs[0, 1].set_title("Train corr on ALL data"); axs[0, 1].set_ylabel("corr"); axs[0, 1].legend(); axs[0, 1].grid(alpha=.3)

axs[1,1].plot(dfk["k"], dfk["mean_test_r_star"], marker="o", label="r λ*")
axs[1,1].plot(dfk["k"], dfk["mean_test_lam0"], marker="s", ls="--", label="r λ=0")
axs[1,1].plot(dfk["k"], dfk["mean_test_mse_star"], marker="o", label="MSE λ*")
axs[1,1].plot(dfk["k"], dfk["mean_test_mse_lam0"], marker="s", ls="--", label="MSE λ=0")
axs[1,1].set_title("Mean test corr vs k"); axs[1,1].set_ylabel("corr"); axs[1,1].legend(); axs[1,1].grid(alpha=.3)

axs[1,0].plot(dfk["k"], dfk["cos_star"], marker="o", label="λ*")
axs[1,0].plot(dfk["k"], dfk["cos_lam0"], marker="s", ls="--", label="λ=0")
axs[1,0].set_title("Cosine similarity vs k"); axs[1,0].set_ylabel(r"cos($\omega_{gt}, \omega_{best}$)"); axs[1,0].grid(alpha=.3); axs[1,0].legend()

axs[1,0].set_xlabel("k (folds; 3 trials per test)"); axs[1,0].legend(); axs[1,0].grid(alpha=.3)
axs[1,1].set_xlabel("k (folds; 3 trials per test)")
axs[1,2].set_xlabel("k (folds; 3 trials per test)")
plt.tight_layout()
plt.show()



=== CV sweep (by k) ===
 k  lam_star  best_lag_ms_star  train_mse_all_star  mean_test_mse_star  train_r_all_star  mean_test_r_star                                                                                                                                                                                                            w_star  cos_star  best_lag_ms_lam0  train_mse_all_lam0  mean_test_mse_lam0  train_r_all_lam0  mean_test_lam0                                                                                                                                                                                                                                                                                    w_lam0  cos_lam0
 3     0.781                 0               0.170               0.192             0.913             0.902 [ 0.06950094  0.04551344 -0.00455896  0.0830288   0.1030812   0.02781512\n -0.04590467  0.01526814  0.05077649  0.01215141 -0.04982555 -0.02634075\n  0.09392

In [ ]:
alphas = np.logspace(-3, 1.5, 15)
print("Alphas:", alphas)

Alphas: [1.00000000e-03 2.09617999e-03 4.39397056e-03 9.21055318e-03
 1.93069773e-02 4.04708995e-02 8.48342898e-02 1.77827941e-01
 3.72759372e-01 7.81370738e-01 1.63789371e+00 3.43332002e+00
 7.19685673e+00 1.50859071e+01 3.16227766e+01]


In [4]:
%matplotlib qt
# %matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
# Use Arial everywhere (falls back if not available)
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size": 12,          # base font size
    "axes.titlesize": 16,     # figure/axes titles
    "axes.labelsize": 13,     # x/y labels
    "legend.fontsize": 11,    # legend text
    "xtick.labelsize": 11,    # tick labels
    "ytick.labelsize": 11,
    "figure.titlesize": 18,   # suptitle
})

In [6]:
import os
import pandas as pd
import matplotlib.pyplot as plt
n_triplets = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 17, 20]

# Folder with your CSVs 
folder = r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\simulated_check\simulation_ridge_reg\sim3"

# The k values you mentioned
k_values = n_triplets
# File name pattern: order_k{K}.csv
for k in k_values:
    path = os.path.join(folder, f"order_ridge_SLOW_k{k}.csv")
    if not os.path.isfile(path):
        #print(f"[skip] File not found: {path}")
        continue

    df = pd.read_csv(path)

    # Ensure numeric & sort by lam
    num_cols = ["lam", "mean_train_mse", "mean_test_mse",
                "mean_train_r", "mean_test_r"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=num_cols).sort_values("lam")

    # ---- find best lambda by test MSE (min) and test r (max) ----
    idx_best_mse = df["mean_test_mse"].idxmin()
    lam_best_mse = df.loc[idx_best_mse, "lam"]

    idx_best_r = df["mean_test_r"].idxmax()
    lam_best_r = df.loc[idx_best_r, "lam"]

    # One figure with two y-axes
    fig, ax1 = plt.subplots()
    ax2 = ax1.twinx()  # second y-axis sharing same x

    # Left axis: correlations
    l1, = ax1.plot(df["lam"], df["mean_train_r"],
                   marker="o", label=r"$\overline{r}_{\text{train}}$")
    l2, = ax1.plot(df["lam"], df["mean_test_r"],
                   marker="s", label=r"$\overline{r}_{\text{test}}$")

    ax1.set_xlabel("λ")
    ax1.set_ylabel("r")
    ax1.set_xscale("log")

    # Right axis: MSE
    l3, = ax2.plot(df["lam"], df["mean_train_mse"],
                   marker="^", linestyle="--", label=r"$\overline{\text{MSE}}_{\text{train}}$")
    l4, = ax2.plot(df["lam"], df["mean_test_mse"],
                   marker="v", linestyle="--", label=r"$\overline{\text{MSE}}_{\text{test}}$")
    ax2.set_ylabel("MSE")

    # ---- vertical lines for best λs ----
    v1 = ax1.axvline(lam_best_mse, linestyle=":", linewidth=1.5,
                     label=r"min $\overline{\text{MSE}}_{\text{test}}$")
    v2 = ax1.axvline(lam_best_r, linestyle="-.", linewidth=1.5,
                     label=r"max $\overline{r}_{\text{test}}$")

    # Title & grid
    fig.suptitle(f"k = {k}")
    ax1.grid(True, linestyle="--", alpha=0.5)

    # Single combined legend
    lines = [l1, l2, l3, l4, v1, v2]
    labels = [ln.get_label() for ln in lines]
    ax1.legend(lines, labels, loc="best")

    fig.tight_layout()


In [7]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter 

folder = r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\simulated_check\simulation_ridge_reg\sim3"
target_ks = [5, 10, 20]

# ---------- First pass: load data and find global ranges ----------
data_by_k = {}
all_r_vals = []
all_mse_vals = []

for k in target_ks:
    path = os.path.join(folder, f"order_ridge_SLOW_k{k}.csv")
    if not os.path.isfile(path):
        print(f"[skip] File not found: {path}")
        continue

    df = pd.read_csv(path)

    for c in ["lam", "mean_train_r", "mean_test_r", "mean_train_mse", "mean_test_mse"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=["lam"]).sort_values("lam")
    data_by_k[k] = df

    all_r_vals.extend(df["mean_train_r"].tolist())
    all_r_vals.extend(df["mean_test_r"].tolist())
    all_mse_vals.extend(df["mean_train_mse"].tolist())
    all_mse_vals.extend(df["mean_test_mse"].tolist())

# global ranges with a small margin
r_min, r_max = min(all_r_vals), max(all_r_vals)
mse_min, mse_max = min(all_mse_vals), max(all_mse_vals)

r_margin = 0.05 * (r_max - r_min) if r_max > r_min else 0.01
mse_margin = 0.05 * (mse_max - mse_min) if mse_max > mse_min else 0.01

r_min_plot = r_min - r_margin
r_max_plot = r_max + r_margin
mse_min_plot = max(0.0, mse_min - mse_margin)  # MSE >= 0
mse_max_plot = mse_max + mse_margin

print("Correlation range:", r_min_plot, r_max_plot)
print("MSE range:", mse_min_plot, mse_max_plot)

# ---------- Second pass: make the figure ----------
fig, axes = plt.subplots(1, len(target_ks), figsize=(10, 3), sharex=False)

all_lines = []
all_labels = []

for j, k in enumerate(target_ks):
    ax = axes[j]
    if k not in data_by_k:
        continue
    df = data_by_k[k]

    lam = df["lam"].values

    # left y-axis: correlations
    ax.set_xscale("log")
    line_r_tr,  = ax.plot(lam, df["mean_train_r"], marker="o", linestyle="-", label=r"$\overline{r}_{\mathrm{train}}$")
    line_r_te,  = ax.plot(lam, df["mean_test_r"], marker="s", linestyle="--", label=r"$\overline{r}_{\mathrm{test}}$")
    ax.set_xlabel(r"$\lambda$")
    if j == 0:
        ax.set_ylabel("Correlation")
    ax.set_title(f"k = {k}")
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.set_ylim(r_min_plot, r_max_plot)

    # *** format correlation ticks with 2 decimals ***
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

    # right y-axis: MSE
    ax2 = ax.twinx()
    line_mse_tr, = ax2.plot(lam, df["mean_train_mse"], marker="^", linestyle="-", label=r"$\overline{\mathrm{MSE}}_{\mathrm{train}}$")
    line_mse_te, = ax2.plot(lam, df["mean_test_mse"], marker="v", linestyle="--", label=r"$\overline{\mathrm{MSE}}_{\mathrm{test}}$")
    if j == 2:
        ax2.set_ylabel("MSE")
    ax2.set_ylim(mse_min_plot, mse_max_plot)

    all_lines.extend([line_r_tr, line_r_te, line_mse_tr, line_mse_te])

# global legend (unique labels, keep order)
seen = set()
uniq_lines = []
uniq_labels = []
for line in all_lines:
    lab = line.get_label()
    if lab not in seen:
        seen.add(lab)
        uniq_lines.append(line)
        uniq_labels.append(lab)

fig.legend(uniq_lines, uniq_labels, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout(rect=[0, 0.05, 1, 1])  # leave room for legend

# optional save
# out_path = os.path.join(folder, "ridge_k4_10_20_twinaxes_same_range.png")
# plt.savefig(out_path, dpi=300)

plt.show()


Correlation range: 0.7964319798502922 0.9188985661370147
MSE range: 0.14312638164018276 0.6414376855654519


In [ ]:
Sxx = (X_tr.T @ X_tr) / len(X_tr)
evals = np.linalg.eigvalsh(Sxx)
gains = evals / (evals + lam)
print(f"cond(Sxx)={evals.max()/max(evals.min(),1e-12):.1e}, "
      f"median_gain={np.median(gains):.3f}, min_gain={gains[0]:.3f}")


NameError: name 'X_tr' is not defined